# Indic GEC (hi/ml/te/ta/bn) with mT5-small 

✨ **OPTIMIZED FOR ENHANCED DATASET:**
- ✅ **Enhanced Dataset**: Uses `enhanced_train.csv` with ~65k synthetic samples
- ✅ **Memory Efficient**: Adafactor optimizer, gradient checkpointing
- ✅ **Robust Training**: Lower LR (3e-4), shuffling, deduplication
- ✅ **Copy Prevention**: Enhanced metrics with copying detection
- ✅ **Task Prefix**: `'grammar correction: '` for proper seq2seq training

This notebook is optimized for training on the comprehensive enhanced dataset
generated with 10 error categories and balanced synthetic variations.

**Required files in the same folder:**
- `enhanced_train.csv` (columns: `incorrect`, `correct`)
- Optional: `dev.csv` (same format). If missing, splits from training data.


## 0. Install dependencies (run once)
If you haven't installed the required libraries, run the cell below.

In [2]:
# If needed, uncomment and run:
# !pip install -U transformers datasets accelerate sentencepiece evaluate tqdm scikit-learn


## 1. Imports, setup, and configuration

In [ ]:
import os
import json
import gc
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import transformers
from transformers import (
    MT5ForConditionalGeneration,
    MT5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed,
)
from transformers.trainer_utils import EvalPrediction
from datasets import Dataset

warnings.filterwarnings('ignore')
SEED = 42
set_seed(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU Memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

# ==================== OPTIMIZED Configuration for Enhanced Dataset ====================
CONFIG: Dict = {
    # Model
    'model_name': 'google/mt5-small',
    'max_input_length': 96,
    'max_target_length': 64,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',

    # Data - UPDATED for enhanced dataset
    'train_file': 'enhanced_train.csv',  # ✅ Using new enhanced dataset (~65k samples)
    'dev_file': 'dev.csv',
    'test_size': 0.1,
    'random_seed': SEED,
    'shuffle_dataset': True,  # ✅ Add explicit shuffling
    'remove_duplicates': True,  # ✅ Remove exact duplicates

    # Training - OPTIMIZED for RTX 3050 6GB + Large Dataset
    'output_dir': './mt5-gec-ml',
    'num_train_epochs': 5,  # Train longer for best accuracy
    'per_device_train_batch_size': 1,  # ✅ Smaller batch for FP16 + large dataset
    'per_device_eval_batch_size': 1,   # ✅ Reduced eval batch size
    'gradient_accumulation_steps': 16,   # ✅ Increased for effective batch size 16
    'learning_rate': 3e-4,  # ✅ Lower LR for large dataset
    'warmup_ratio': 0.15,   # ✅ More warmup for stability
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'fp16': False,  # ✅ FIXED: Disable FP16 due to Adafactor compatibility issue
    'gradient_checkpointing': True,
    'optim': 'adamw_torch',  # ✅ FIXED: Use AdamW instead of Adafactor for FP16 compatibility

    # Evaluation / saving
    'evaluation_strategy': 'epoch',
    'save_strategy': 'epoch',
    'logging_steps': 50,
    'save_total_limit': 2,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'gleu',
    'greater_is_better': True,
    'early_stopping_patience': 8,  # Increased patience for better convergence

    # Generation
    'generation_config': {
        'max_length': 128,
        'num_beams': 4,
        'early_stopping': True,
        'repetition_penalty': 1.2,
        'no_repeat_ngram_size': 3,
        'length_penalty': 1.0,
        'do_sample': False,
    }
}
# Language configuration: select among hi, ml, te, ta, bn
LANGS = {'hi': 'Hindi','ml': 'Malayalam','te': 'Telugu','ta': 'Tamil','bn': 'Bengali'}
CONFIG['lang_code'] = os.environ.get('GEC_LANG', 'ml').lower()
if CONFIG['lang_code'] not in LANGS:
    CONFIG['lang_code'] = 'ml'
CONFIG['lang_name'] = LANGS[CONFIG['lang_code']]
CONFIG['output_dir'] = f"./mt5-gec-{CONFIG['lang_code']}"

# Optionally pick per-language files if present
_train_lang = f"enhanced_train_{CONFIG['lang_code']}.csv"
if Path(_train_lang).exists():
    CONFIG['train_file'] = _train_lang
_dev_lang = f"dev_{CONFIG['lang_code']}.csv"
if Path(_dev_lang).exists():
    CONFIG['dev_file'] = _dev_lang

CONFIG

## 2. Data loading and cleaning - WITH AUGMENTATION

In [1]:
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ''
    text = str(text).strip()
    text = ' '.join(text.split())
    text = ''.join(ch for ch in text if ord(ch) >= 32 or ch == '\n')
    return text

def load_and_prepare_data(config: Dict):
    train_path = Path(config['train_file'])
    if not train_path.exists():
        raise FileNotFoundError(f'Training file not found: {train_path}')

    train_df = pd.read_csv(train_path, encoding='utf-8')
    print(f'✅ Loaded enhanced dataset with {len(train_df)} samples')
    
    # Determine columns (support both formats)
    if 'input' in train_df.columns and 'output' in train_df.columns:
        input_col, output_col = 'input', 'output'
    elif 'incorrect' in train_df.columns and 'correct' in train_df.columns:
        input_col, output_col = 'incorrect', 'correct'  # ✅ Support enhanced dataset format
    else:
        input_col, output_col = train_df.columns[0], train_df.columns[1]

    train_df = train_df[[input_col, output_col]].copy()
    train_df.columns = ['input_text', 'output_text']
    train_df['input_text'] = train_df['input_text'].apply(clean_text)
    train_df['output_text'] = train_df['output_text'].apply(clean_text)
    train_df = train_df[(train_df['input_text'] != '') & (train_df['output_text'] != '')]
    train_df = train_df[(train_df['input_text'].str.len().between(5, 200)) & (train_df['output_text'].str.len().between(5, 200))]

    print(f'After cleaning: {len(train_df)} samples')
    
    # ✅ CRITICAL: Remove exact duplicates to prevent overfitting
    if config.get('remove_duplicates', True):
        before_dedup = len(train_df)
        train_df = train_df.drop_duplicates(subset=['input_text', 'output_text'], keep='first')
        after_dedup = len(train_df)
        print(f'🧹 Removed {before_dedup - after_dedup} duplicate pairs')
    
    # ✅ CRITICAL: Shuffle dataset for better training
    if config.get('shuffle_dataset', True):
        train_df = train_df.sample(frac=1.0, random_state=config['random_seed']).reset_index(drop=True)
        print('🔀 Dataset shuffled')

    print(f'Final training samples: {len(train_df)}')

    # 🗺️ Enhanced dataset already contains comprehensive synthetic data
    # No need for additional augmentation - focus on dev set preparation
    
    dev_path = Path(config['dev_file'])
    if dev_path.exists():
        dev_df = pd.read_csv(dev_path, encoding='utf-8')
        # Determine dev columns
        if 'input' in dev_df.columns and 'output' in dev_df.columns:
            dev_input_col, dev_output_col = 'input', 'output'
        elif 'incorrect' in dev_df.columns and 'correct' in dev_df.columns:
            dev_input_col, dev_output_col = 'incorrect', 'correct'
        else:
            dev_input_col, dev_output_col = dev_df.columns[0], dev_df.columns[1]
            
        dev_df = dev_df[[dev_input_col, dev_output_col]].copy()
        dev_df.columns = ['input_text', 'output_text']
        dev_df['input_text'] = dev_df['input_text'].apply(clean_text)
        dev_df['output_text'] = dev_df['output_text'].apply(clean_text)
        dev_df = dev_df[(dev_df['input_text'] != '') & (dev_df['output_text'] != '')]
        print(f'Dev set loaded: {len(dev_df)} samples')
    else:
        # Split from training data if no dev file
        print('📊 Splitting dev set from training data...')
        train_df, dev_df = train_test_split(
            train_df, 
            test_size=config['test_size'], 
            random_state=config['random_seed'],
            stratify=None  # Can't stratify with such diverse synthetic data
        )
        print(f'Created dev split: {len(dev_df)} samples')

    # ✅ Calculate important statistics for enhanced dataset
    identical_train = int((train_df['input_text'] == train_df['output_text']).sum())
    identical_dev = int((dev_df['input_text'] == dev_df['output_text']).sum())
    
    print('\n📊 Dataset Statistics:')
    print(f'   • Train samples: {len(train_df):,}')
    print(f'   • Dev samples: {len(dev_df):,}')
    print(f'   • Identity pairs in train: {identical_train} ({identical_train/len(train_df)*100:.1f}%)')
    print(f'   • Identity pairs in dev: {identical_dev} ({identical_dev/len(dev_df)*100:.1f}%)')
    
    # Show sample corrections from enhanced dataset
    print('\n📝 Sample corrections:')
    correction_samples = train_df[train_df['input_text'] != train_df['output_text']].head(3)
    for i, (_, row) in enumerate(correction_samples.iterrows(), 1):
        print(f"{i}. Input:  {row['input_text'][:80]}")
        print(f"   Output: {row['output_text'][:80]}")
    
    return train_df, dev_df

train_df, dev_df = load_and_prepare_data(CONFIG)
len(train_df), len(dev_df)

NameError: name 'Dict' is not defined

## 3. Load tokenizer and model

In [ ]:
def load_model_and_tokenizer(config: Dict):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    tokenizer = MT5Tokenizer.from_pretrained(config['model_name'])
    model = MT5ForConditionalGeneration.from_pretrained(
        config['model_name'],
        torch_dtype=(torch.float16 if config['fp16'] else torch.float32),
    )
    # Ensure embeddings are tied for correctness
    model.config.tie_word_embeddings = True
    model.tie_weights()
    if config.get('gradient_checkpointing', False):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.config.use_cache = False
    model = model.to(config['device'])
    print('Vocab size:', len(tokenizer))
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'Model params: {total_params:.1f}M')
    if torch.cuda.is_available():
        print('GPU mem allocated (GB):', round(torch.cuda.memory_allocated() / 1024**3, 2))
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer(CONFIG)

## 4. Tokenization and dataset preparation - FIXED TASK PREFIX

In [ ]:
def create_tokenization_function(tokenizer, config: Dict):
    def tokenize_function(examples):
        # Use a language-agnostic task prefix for seq2seq correction
        # This prevents the model from treating it as an infilling task
        inputs = ['grammar correction: ' + text for text in examples['input_text']]
        targets = examples['output_text']
        model_inputs = tokenizer(
            inputs,
            max_length=config['max_input_length'],
            truncation=True,
            padding=False,
        )
        labels = tokenizer(
            text_target=targets,
            max_length=config['max_target_length'],
            truncation=True,
            padding=False,
        )
        model_inputs['labels'] = labels['input_ids']
        return model_inputs
    return tokenize_function

tokenize_function = create_tokenization_function(tokenizer, CONFIG)

hf_train = Dataset.from_pandas(train_df)
hf_dev = Dataset.from_pandas(dev_df)

tokenized_train = hf_train.map(tokenize_function, batched=True, remove_columns=hf_train.column_names, desc='Tokenizing train')
tokenized_dev = hf_dev.map(tokenize_function, batched=True, remove_columns=hf_dev.column_names, desc='Tokenizing dev')

print('Tokenized sizes:', len(tokenized_train), len(tokenized_dev))

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,  # dynamic padding
)

## 5. Metrics (GLEU proxy)

In [ ]:
def compute_metrics(eval_preds: EvalPrediction):
    """Enhanced metrics with copying detection and better error handling"""
    # Support both EvalPrediction and (predictions, labels) tuple
    if isinstance(eval_preds, tuple):
        predictions, labels = eval_preds
    else:
        predictions, labels = eval_preds.predictions, eval_preds.label_ids
    
    # Unwrap predictions if generate() returns a tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # ✅ Robust prediction handling
    preds = np.array(predictions)
    if preds.ndim == 3:  # logits -> ids
        preds = preds.argmax(-1)
    preds = preds.astype(np.int64, copy=False)
    
    # ✅ Guard against invalid token ids
    vocab_size = len(tokenizer)
    preds = np.where((preds >= 0) & (preds < vocab_size), preds, tokenizer.pad_token_id)

    # ✅ Decode predictions and labels
    try:
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        # Replace -100 to decode labels
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    except Exception as e:
        print(f'Warning: Error in decoding: {e}')
        return {'gleu': 0.0, 'exact_match': 0.0, 'copy_rate': 100.0}

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # ✅ Calculate GLEU-like proxy using token F1
    gleu_scores = []
    exact_matches = 0
    copy_detections = 0  # ✅ Track copying behavior
    
    for pred, ref in zip(decoded_preds, decoded_labels):
        # Check for exact match
        if pred == ref:
            exact_matches += 1
            
        # ✅ Detect potential copying (suspicious patterns)
        if '<extra_id_' in pred or len(pred) == 0:
            copy_detections += 1
            
        # Calculate token-level F1
        pt = set(pred.lower().split())
        rt = set(ref.lower().split())
        if not rt:
            gleu_scores.append(0.0)
            continue
            
        overlap = pt & rt
        precision = len(overlap) / len(pt) if pt else 0.0
        recall = len(overlap) / len(rt)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        gleu_scores.append(f1)
    
    # ✅ Calculate comprehensive metrics
    gleu = float(np.mean(gleu_scores) * 100) if gleu_scores else 0.0
    exact_match_rate = exact_matches / len(decoded_preds) * 100 if decoded_preds else 0.0
    copy_rate = copy_detections / len(decoded_preds) * 100 if decoded_preds else 0.0

    return {
        'gleu': gleu,
        'exact_match': exact_match_rate,
        'copy_rate': copy_rate
    }

## 6. Training - WITH IMPROVED SETTINGS

In [ ]:
# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_eval_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_ratio=CONFIG['warmup_ratio'],
    label_smoothing_factor=0.1,
    lr_scheduler_type='cosine',
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    fp16=CONFIG['fp16'],  # CRITICAL: Must match checkpoint
    optim=CONFIG['optim'],
    eval_strategy=CONFIG['evaluation_strategy'],
    save_strategy=CONFIG['save_strategy'],
    logging_steps=CONFIG['logging_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    load_best_model_at_end=CONFIG['load_best_model_at_end'],
    metric_for_best_model=CONFIG['metric_for_best_model'],
    greater_is_better=CONFIG['greater_is_better'],
    predict_with_generate=True,
    generation_max_length=CONFIG['generation_config']['max_length'],
    generation_num_beams=1,
    eval_accumulation_steps=1,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    remove_unused_columns=True,
    report_to='none',
    push_to_hub=False,
    seed=CONFIG['random_seed'],
    data_seed=CONFIG['random_seed'],
)

# Recreate trainer with correct configuration
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG['early_stopping_patience'])],
)

print("✅ Trainer recreated with FP16 + AdamW compatibility")

print("🚀 Starting training...")
_ = trainer.train()

# Save final model
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print('✅ Training completed and model saved!')

## 7. Inference — generate predictions and save CSV - FIXED

In [ ]:
def generate_predictions(model_path: str, test_file: str, output_file: str = 'predictions.csv', batch_size: int = 4):
    print(f'Loading model from {model_path} ...')
    tok = MT5Tokenizer.from_pretrained(model_path)
    mdl = MT5ForConditionalGeneration.from_pretrained(model_path).to(device)
    mdl.eval()

    df = pd.read_csv(test_file, encoding='utf-8')
    if 'input' in df.columns:
        input_col = 'input'
    else:
        input_col = df.columns[0]
    df = df[[input_col]].copy()
    df.columns = ['input_text']
    df['input_text'] = df['input_text'].apply(clean_text)
    df = df[df['input_text'] != '']

    preds: List[str] = []
    with torch.no_grad():
        for i in tqdm(range(0, len(df), batch_size), desc='Generating'):
            batch = df.iloc[i:i+batch_size]
            # ✅ CRITICAL FIX: Use same task prefix as training
            inputs = ['grammar correction: ' + s for s in batch['input_text'].tolist()]
            enc = tok(
                inputs,
                max_length=CONFIG['max_input_length'],
                truncation=True,
                padding=True,
                return_tensors='pt',
            ).to(device)
            outputs = mdl.generate(
                **enc,
                max_length=CONFIG['generation_config']['max_length'],
                num_beams=CONFIG['generation_config']['num_beams'],
                early_stopping=CONFIG['generation_config']['early_stopping'],
                repetition_penalty=CONFIG['generation_config']['repetition_penalty'],
                no_repeat_ngram_size=CONFIG['generation_config']['no_repeat_ngram_size'],
            )
            decoded = tok.batch_decode(outputs, skip_special_tokens=True)
            preds.extend(decoded)
            if torch.cuda.is_available() and i % 100 == 0:
                torch.cuda.empty_cache()
                gc.collect()

    out_df = pd.DataFrame({
        'Input sentence': df['input_text'].tolist()[:len(preds)],
        'Output sentence': preds,
    })
    out_df.to_csv(output_file, index=False, encoding='utf-8')
    print('Predictions saved to', output_file)
    print(out_df.head())
    return out_df

# Generate predictions
pred_file = f"predictions_{CONFIG['lang_code']}.csv"
_ = generate_predictions(CONFIG['output_dir'], CONFIG['dev_file'], pred_file)

## 8. Evaluation - Check if fixes worked

In [ ]:
def gleu_proxy(preds, refs):
    scores = []
    for pred, ref in zip(preds, refs):
        pt, rt = set(str(pred).lower().split()), set(str(ref).lower().split())
        if not rt:
            scores.append(0.0)
            continue
        overlap = pt & rt
        p = len(overlap) / len(pt) if pt else 0.0
        r = len(overlap) / len(rt)
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        scores.append(f1)
    return float(np.mean(scores) * 100)

# Load references
dev_df_eval = pd.read_csv(CONFIG['dev_file'], encoding='utf-8')
in_col = 'input' if 'input' in dev_df_eval.columns else dev_df_eval.columns[0]
ref_col = 'output' if 'output' in dev_df_eval.columns else dev_df_eval.columns[1]
refs = dev_df_eval[ref_col].astype(str).tolist()

# Load model predictions
pred_file = f"predictions_{CONFIG['lang_code']}.csv"
pred_df = pd.read_csv(pred_file, encoding='utf-8')
preds = pred_df['Output sentence'].astype(str).tolist()

# Compute metrics
gleu_model = gleu_proxy(preds, refs)
gleu_identity = gleu_proxy(dev_df_eval[in_col].astype(str).tolist(), refs)
exact_match = (pd.Series(preds) == pd.Series(refs)).mean() * 100.0

print("🔥 RESULTS AFTER FIXES:")
print(f"Dev GLEU (proxy) — FIXED model: {gleu_model:.2f}")
print(f"Dev GLEU (proxy) — identity baseline: {gleu_identity:.2f}")
print(f"Exact match rate: {exact_match:.2f}%")

# Check for <extra_id_0> tokens
extra_id_count = sum(1 for pred in preds if '<extra_id_0>' in str(pred))
print(f"\n🚨 Sentences with <extra_id_0>: {extra_id_count}/{len(preds)} ({extra_id_count/len(preds)*100:.1f}%)")

if gleu_model > gleu_identity:
    print("\n🎉 SUCCESS! Model is now better than identity baseline!")
else:
    print("\n⚠️  Model still below identity baseline. May need more training or different approach.")

# Show sample results
print("\n📋 Sample results:")
for i in range(min(5, len(pred_df))):
    print(f"{i+1}.")
    print("Input:   ", pred_df.iloc[i]['Input sentence'][:100])
    print("Pred:    ", pred_df.iloc[i]['Output sentence'][:100])
    print("Ref:     ", dev_df_eval.iloc[i][ref_col][:100])
    print()

In [ ]:
# Inference cell: generate predictions for test.csv and save to predictions_test.csv
# It will install missing packages if necessary, load the local model in ./mt5-gec-ml,
# detect the input column in test.csv, run batched generation, and save the outputs.

import os
import sys
import subprocess

def _install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + pkgs)

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    import pandas as pd
except Exception:
    print("Installing required packages: transformers, sentencepiece, safetensors, torch, pandas")
    _install(["transformers==4.37.0", "sentencepiece", "safetensors", "torch", "pandas"])
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    import pandas as pd

# Paths / configuration
MODEL_DIR = "./mt5-gec-ml"
TEST_CSV = "test.csv"
OUT_CSV = "predictions_test.csv"
BATCH_SIZE = 8
MAX_LENGTH = 128
NUM_BEAMS = 4

# Validate files
if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(f"Model directory not found: {MODEL_DIR}")
if not os.path.exists(TEST_CSV):
    raise FileNotFoundError(f"Test file not found: {TEST_CSV}")

# Load tokenizer + model
print("Loading tokenizer and model from", MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Read test data
print("Reading", TEST_CSV)
df = pd.read_csv(TEST_CSV)
if df.shape[0] == 0:
    raise ValueError("Test CSV is empty")

# Heuristic to find the input column
candidate_cols = [
    "input_sentence", "input", "text", "sentence", "source", "src",
    "source_sentence", "orig", "sentence_text", "incorrect"
]
input_col = None
for c in candidate_cols:
    if c in df.columns:
        input_col = c
        break
if input_col is None:
    # fallback to first column
    input_col = df.columns[0]

print(f"Using input column: '{input_col}' (columns: {list(df.columns)})")

# Ensure text is str and fillna
inputs_list = df[input_col].fillna("").astype(str).tolist()

# Batched generation
outputs = []
for i in range(0, len(inputs_list), BATCH_SIZE):
    batch = inputs_list[i : i + BATCH_SIZE]
    enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        gen_ids = model.generate(
            **enc,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            do_sample=False
        )
    decoded = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)
    outputs.extend(decoded)

# Safety: if generation failed for any item, fill with empty string
if len(outputs) != len(inputs_list):
    # pad or trim
    outputs = (outputs + [""] * len(inputs_list))[: len(inputs_list)]

# Save results
out_df = pd.DataFrame({
    "input_sentence": inputs_list,
    "output_sentence": outputs
})
out_df.to_csv(OUT_CSV, index=False)
print(f"Saved predictions to {OUT_CSV}. Rows: {len(out_df)}")

# Display a sample
out_df.head(10)


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MT5Tokenizer'. 
The class this function is called from is 'T5Tokenizer'.


Loading tokenizer and model from ./mt5-gec-ml


You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Reading test.csv
Using input column: 'Input sentence' (columns: ['Input sentence'])
